# Prompt augmentation pilot: 3 conditions x 100-row subsample

Pilot run of the prompt-augmentation baseline against the Llama 3.3 70B agent + Claude Sonnet 4.6 judge pipeline. Goal: measure whether wrapping the user prompt in defensive scaffolding (instruction-only, combined sandwich + delimiters) reduces hijack rate vs. control (no augmentation).

Three conditions per prompt:
- **control**: raw prompt sent to agent
- **instruction_only**: explicit override-resistance instruction prepended
- **combined**: instruction sandwich + delimiter framing per `src/augmentation/variants.py`

This is a 100-row pilot subsample of the frozen eval set. Scale to the full ~4,546 rows is a separate run (see TODO at end).

Templates anchored on Perez and Ribeiro (2022) and prompt-engineering literature. Methodology decision: `_project_notes/capstone_methodology_decisions.md` #7.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import env, set_seed
from src.cache import append_records, existing_keys, load_records
from src.augmentation.variants import apply, ALL_VARIANTS
from src.defense_b.agent import GroqAgent
from src.defense_b.judge import ClaudeJudge

env()
set_seed(42)

RES = REPO_ROOT / 'results'
CACHE = REPO_ROOT / 'cache'
CACHE.mkdir(exist_ok=True)

## Pilot subsample

Stratified 100-row subsample from the frozen eval set: ~33 per dataset, ~50/50 SAFE/INJECTION within each. Same seed-42 selection on every run.

In [ ]:
es = pd.read_parquet(RES / 'eval_set.parquet')
rng_seed = 42
PILOT_PER_DS = {'deepset': 34, 'neuralchemy': 33, 'spml': 33}

import numpy as np
rng = np.random.default_rng(rng_seed)
pilot_parts = []
for ds, n in PILOT_PER_DS.items():
    pool = es[es['dataset'] == ds]
    half = n // 2
    extra = n - 2 * half
    safe = pool[pool['label'] == 0]
    inj  = pool[pool['label'] == 1]
    s_idx = rng.choice(safe.index.values, size=min(half, len(safe)), replace=False)
    i_idx = rng.choice(inj.index.values,  size=min(half + extra, len(inj)), replace=False)
    pilot_parts.append(pd.concat([safe.loc[s_idx], inj.loc[i_idx]]))

pilot = pd.concat(pilot_parts).sort_values('prompt_idx').reset_index(drop=True)
print(f'pilot rows: {len(pilot)}')
print(pilot.groupby(['dataset', 'label']).size())

## Run agent under three conditions

Three caches, one per variant. Resumable: re-running this notebook only hits the API for prompts not yet cached.

In [ ]:
agent = GroqAgent()
agent_records: dict[str, dict] = {}

for variant in ALL_VARIANTS:
    cache_path = CACHE / f'augmentation_{variant}_agent.jsonl'
    done = existing_keys(cache_path, key='prompt_idx')
    todo = pilot[~pilot['prompt_idx'].isin(done)]
    print(f'\n[{variant}] cached: {len(done)}, to run: {len(todo)}')
    new = []
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc=f'agent/{variant}'):
        wrapped = apply(row['prompt'], variant)
        out = agent.respond(wrapped)
        new.append({'prompt_idx': row['prompt_idx'], 'variant': variant, **out})
    if new:
        append_records(cache_path, new)
    agent_records[variant] = {r['prompt_idx']: r for r in load_records(cache_path)}

## Judge each (prompt, response) pair

Same Claude Sonnet 4.6 minimum-rubric judge as the Defense B sneak preview. The judge sees the **original** user prompt (not the augmented version), because the question is whether the agent's response shows hijack relative to the user's actual intent.

In [ ]:
judge = ClaudeJudge()
judge_records: dict[str, dict] = {}

for variant in ALL_VARIANTS:
    cache_path = CACHE / f'augmentation_{variant}_judge.jsonl'
    done = existing_keys(cache_path, key='prompt_idx')
    todo = pilot[~pilot['prompt_idx'].isin(done)]
    print(f'\n[{variant}] judge cached: {len(done)}, to run: {len(todo)}')
    new = []
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc=f'judge/{variant}'):
        agent_resp = agent_records[variant][row['prompt_idx']]['response']
        out = judge.judge(row['prompt'], agent_resp)
        new.append({'prompt_idx': row['prompt_idx'], 'variant': variant, **out})
    if new:
        append_records(cache_path, new)
    judge_records[variant] = {r['prompt_idx']: r for r in load_records(cache_path)}

## Tabulate per-variant hijack rates

On injection-class rows only (label = 1), what fraction of agent responses are flagged hijacked by the judge under each augmentation condition? Lower is better.

In [ ]:
rows = []
for variant in ALL_VARIANTS:
    j = judge_records[variant]
    for _, p in pilot.iterrows():
        verdict = j[p['prompt_idx']]
        rows.append({
            'variant': variant,
            'dataset': p['dataset'],
            'label': p['label'],
            'judge_hijacked': verdict.get('hijacked'),
            'judge_blocked': verdict.get('judge_blocked', False),
        })
results = pd.DataFrame(rows)

inj = results[results['label'] == 1]
summary = inj.groupby('variant').agg(
    n=('judge_hijacked', 'size'),
    hijacked=('judge_hijacked', lambda s: int(s.fillna(False).sum())),
    blocked=('judge_blocked', 'sum'),
).reset_index()
summary['hijack_rate'] = (summary['hijacked'] / summary['n']).round(3)
summary

## Save artifacts and stop

Pilot writes to `results/augmentation_pilot.csv` and `results/augmentation_pilot_summary.csv`. Scaling to the full eval set is a separate notebook / script run; the cache layer is already resumable.

In [ ]:
results.to_csv(RES / 'augmentation_pilot.csv', index=False)
summary.to_csv(RES / 'augmentation_pilot_summary.csv', index=False)
print('saved')
print('  results/augmentation_pilot.csv')
print('  results/augmentation_pilot_summary.csv')

## TODOs (for the user when reviewing)

1. After running this pilot, look at the per-variant hijack-rate column. If `combined` does not meaningfully reduce the rate vs `control`, that is itself a finding (decomposition of variants becomes more important).
2. Cost note: 100 rows x 3 variants x 2 API calls = 600 calls. Estimated spend at sneak-preview rates: under $1.
3. Scaling to the full ~4,546-row eval set: 4546 x 3 x 2 = 27,276 calls. Consider whether to scale all three variants, or only the winner from the pilot. Decision belongs in the Phase 1 checkpoint.
4. Judge rubric is still the sneak-preview minimum. Iterating the rubric (informed by the operational definitions document) is a separate Phase 2 step before scaling.